# SOLIDER Swin-Small partial fine-tune for Warehouse cameras

Companion to `docs/superpowers/specs/2026-05-28-reid-fine-tune-design.md`.

**Goal:** push back-cam (0394 / 0395 / 0396) cross-camera ReID margin above the MCPT `sim_th = 0.85` gate so that MCT links 5+ global IDs per back camera (currently 0 / 1 / 1).

**Strategy:** partial fine-tune (Swin stage index 3 + BN-neck + ID head). Stages 0-2 frozen. SOLIDER's own ID + triplet + center loss recipe. 20-30 epochs cosine schedule. Leave scene_044 out for eval.

**Recommended runtime:** Colab Pro V100 / A100. Free T4 works (3x slower).

All hand-off-checklist steps live in this notebook. Run cells top to bottom.

## Phase 1 — Environment & GPU check

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'GPU runtime required. Runtime > Change runtime type > GPU.'

In [ ]:
import os, sys
WORKDIR = '/content'
os.chdir(WORKDIR)

if not os.path.exists('aic24-nvidia'):
    !git clone --depth 1 https://github.com/harshbhatt-awl/aic24-nvidia.git

REPO = '/content/aic24-nvidia'
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# Minimal deps for SOLIDER backbone + training. Colab already has torch, torchvision, numpy, pandas.
!pip install -q timm einops gdown opencv-python-headless ftfy tqdm scikit-learn
print('cwd:', os.getcwd())

In [ ]:
# SOLIDER base checkpoint (Google Drive ID lives in aic24_nvidia/models/solider/__init__.py).
os.makedirs('weights', exist_ok=True)
if not os.path.exists('weights/solider_swin_small.pth'):
    !gdown 1C-aIZdFyjFsZX4W4feG-Ex39RU2Qvu3b -O weights/solider_swin_small.pth
!ls -lh weights/

In [ ]:
# Smoke-test the SOLIDER wrapper end-to-end before we bother with data.
from aic24_nvidia.models.solider import load_solider_swin_small, SOLIDER_SIZE, SOLIDER_MEAN, SOLIDER_STD
import torch

_model = load_solider_swin_small('weights/solider_swin_small.pth').cuda()
with torch.no_grad():
    _x = torch.randn(2, 3, *SOLIDER_SIZE).cuda()
    _f = _model(_x)
print('feat shape:', tuple(_f.shape), '| expected (2, 768)')
del _model, _x, _f; torch.cuda.empty_cache()

## Phase 2 — Data sourcing

Pick one of the options below (see spec §Data sourcing). All produce a directory tree like:

```
<DATA_ROOT>/
  train/scene_NNN/{videos/camera_NNNN.mp4, ground_truth.json, calibration.json}
  val/scene_NNN/...
```

**Recommended (Option A):** upload the entire `MTMC_Tracking_2024/` you have locally to Google Drive once, then mount Drive. Scene_044 alone gives single-scene training (acceptable but suboptimal — see Decision Rule).

**Option B:** pull additional scenes from a HuggingFace mirror (verify format matches before relying on it).

**Option C:** fall back to scene_044 only with heavy augmentation (lowest ceiling).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# >>> EDIT THIS to point at your Drive copy of MTMC_Tracking_2024 <<<
DRIVE_DATA_ROOT = '/content/drive/MyDrive/aic24_data/MTMC_Tracking_2024'
LOCAL_DATA_ROOT = '/content/data/MTMC_Tracking_2024'

os.makedirs('/content/data', exist_ok=True)
!ln -sfn "$DRIVE_DATA_ROOT" "$LOCAL_DATA_ROOT"
!ls "$LOCAL_DATA_ROOT"

## Phase 3 — Build crops + training manifest

For every (camera, frame, person) annotation in each training scene, decode the frame, crop the bbox with a small padding, save JPEG. Subsample frames (every Nth) to keep the dataset manageable.

Scene_044 is held out — it serves both as ReID val and the downstream pipeline test.

In [ ]:
from pathlib import Path
import json, csv
import cv2
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm

DATA_ROOT = Path(LOCAL_DATA_ROOT)
CROPS_ROOT = Path('/content/crops')
CROPS_ROOT.mkdir(parents=True, exist_ok=True)

HOLDOUT_SCENE = 'scene_044'
FRAME_STEP = 5     # take every Nth frame
BBOX_PAD = 8       # px padding around bbox
MIN_BBOX_PX = 32   # skip tiny boxes
JPEG_Q = 92

def discover_scenes(root: Path):
    scenes = []
    for split in ('train', 'val'):
        d = root / split
        if not d.exists():
            continue
        for sc in sorted(d.iterdir()):
            if sc.is_dir() and (sc / 'ground_truth.json').exists():
                scenes.append((split, sc))
    return scenes

scenes = discover_scenes(DATA_ROOT)
print(f'Found {len(scenes)} scenes:')
for split, sc in scenes:
    held = ' [HELD OUT]' if sc.name == HOLDOUT_SCENE else ''
    print(f'  {split}/{sc.name}{held}')

In [ ]:
def parse_gt(gt_path: Path):
    """Yield (camera, frame, person_id, bbox_xywh).

    Handles the NVIDIA MTMC 2024 schema documented in our adapter:
      {cameras:{cam:{K,R,t}}, annotations:[{camera, frame, person_id, bbox_2d, world_xy}]}
    bbox_2d is [x, y, w, h].
    """
    with gt_path.open() as f:
        gt = json.load(f)
    for ann in gt.get('annotations', []):
        bb = ann.get('bbox_2d') or ann.get('bbox')
        if bb is None:
            continue
        if len(bb) == 4:
            x, y, w, h = bb
        else:
            continue
        yield ann['camera'], int(ann['frame']), int(ann['person_id']), (float(x), float(y), float(w), float(h))

def extract_crops(scene_dir: Path, out_root: Path, frame_step: int):
    """Decode frames lazily — load each video once, walk annotations grouped by camera."""
    gt = scene_dir / 'ground_truth.json'
    if not gt.exists():
        return []
    by_cam_frame = defaultdict(list)
    for cam, fr, pid, bb in parse_gt(gt):
        if fr % frame_step != 0:
            continue
        by_cam_frame[(cam, fr)].append((pid, bb))

    rows = []
    cams = sorted({c for c, _ in by_cam_frame})
    for cam in cams:
        vid = scene_dir / 'videos' / f'{cam}.mp4'
        if not vid.exists():
            print(f'  missing video: {vid}')
            continue
        cap = cv2.VideoCapture(str(vid))
        if not cap.isOpened():
            print(f'  cannot open: {vid}')
            continue
        try:
            frames_needed = sorted(fr for (c, fr) in by_cam_frame if c == cam)
            for fr in frames_needed:
                cap.set(cv2.CAP_PROP_POS_FRAMES, fr)
                ok, img = cap.read()
                if not ok or img is None:
                    continue
                H, W = img.shape[:2]
                for pid, (x, y, w, h) in by_cam_frame[(cam, fr)]:
                    if w < MIN_BBOX_PX or h < MIN_BBOX_PX:
                        continue
                    x0 = max(0, int(x) - BBOX_PAD)
                    y0 = max(0, int(y) - BBOX_PAD)
                    x1 = min(W, int(x + w) + BBOX_PAD)
                    y1 = min(H, int(y + h) + BBOX_PAD)
                    crop = img[y0:y1, x0:x1]
                    if crop.size == 0:
                        continue
                    out_dir = out_root / scene_dir.name / cam
                    out_dir.mkdir(parents=True, exist_ok=True)
                    out_path = out_dir / f'pid{pid:04d}_f{fr:06d}.jpg'
                    cv2.imwrite(str(out_path), crop, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_Q])
                    rows.append((scene_dir.name, cam, fr, pid, str(out_path)))
        finally:
            cap.release()
    return rows

manifest_rows = []
for split, sc in tqdm(scenes, desc='scenes'):
    rows = extract_crops(sc, CROPS_ROOT, FRAME_STEP)
    print(f'  {sc.name}: {len(rows)} crops')
    manifest_rows.extend(rows)

MANIFEST = CROPS_ROOT / 'manifest.csv'
with MANIFEST.open('w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['scene', 'camera', 'frame', 'person_id', 'crop_path'])
    w.writerows(manifest_rows)
print(f'\nTotal crops: {len(manifest_rows)}')
print(f'Manifest: {MANIFEST}')

In [ ]:
# Sanity: counts by scene / camera / unique pid; verify train/val split is non-empty.
import pandas as pd
df = pd.read_csv(MANIFEST)
df['split'] = df['scene'].apply(lambda s: 'val' if s == HOLDOUT_SCENE else 'train')
print(df.groupby(['split', 'scene']).agg(crops=('crop_path', 'size'), ids=('person_id', 'nunique'), cams=('camera', 'nunique')))
n_train = df[df.split=='train'].person_id.nunique()
n_val = df[df.split=='val'].person_id.nunique()
print(f'\nTrain unique IDs: {n_train}')
print(f'Val unique IDs:   {n_val}')
print(f'Val cameras:      {sorted(df[df.split=="val"].camera.unique())}')
if n_train == 0:
    print('\n!!! TRAIN SPLIT IS EMPTY !!!')
    print(f'Your data root only contains the held-out scene ({HOLDOUT_SCENE!r}).')
    print('Upload more scene_NNN/ directories to your Drive folder, then re-run Phase 3.')
    print('Phase 4 will raise RuntimeError if you proceed without fixing this.')

## Phase 4 — Dataset / dataloader

Identity-balanced sampler: each batch has P identities × K instances per identity. Required for the triplet loss to find hard anchors/positives/negatives in-batch.

Augmentation matches SOLIDER conventions: hflip, color jitter, random erasing. Resize is fixed at 384×128 — must match inference preprocessing in `aic24_nvidia/models/solider/__init__.py`.

In [ ]:
import random
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from PIL import Image
from collections import defaultdict

from aic24_nvidia.models.solider import SOLIDER_SIZE, SOLIDER_MEAN, SOLIDER_STD

TRAIN_TF = transforms.Compose([
    transforms.Resize(SOLIDER_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=SOLIDER_MEAN, std=SOLIDER_STD),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.4), value=0),
])
EVAL_TF = transforms.Compose([
    transforms.Resize(SOLIDER_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=SOLIDER_MEAN, std=SOLIDER_STD),
])

class ReIDCropDataset(Dataset):
    def __init__(self, frame, transform):
        # Re-index person_id within this dataset to a contiguous [0, K).
        self.paths = frame['crop_path'].tolist()
        self.cams = frame['camera'].tolist()
        raw_pids = frame['person_id'].tolist()
        unique = sorted(set(raw_pids))
        self.pid_to_idx = {p: i for i, p in enumerate(unique)}
        self.labels = [self.pid_to_idx[p] for p in raw_pids]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        x = self.transform(img)
        return x, self.labels[idx], self.cams[idx]

class PKSampler(Sampler):
    """Each batch: P identities x K instances. Drops identities with < K samples."""
    def __init__(self, labels, P=8, K=4, num_batches=None):
        self.P, self.K = P, K
        self.label_to_idx = defaultdict(list)
        for i, y in enumerate(labels):
            self.label_to_idx[y].append(i)
        self.label_to_idx = {y: ix for y, ix in self.label_to_idx.items() if len(ix) >= K}
        self.labels = list(self.label_to_idx.keys())
        if num_batches is None:
            num_batches = max(1, len(labels) // (P * K))
        self.num_batches = num_batches

    def __iter__(self):
        for _ in range(self.num_batches):
            chosen = random.sample(self.labels, self.P)
            batch = []
            for y in chosen:
                batch.extend(random.sample(self.label_to_idx[y], self.K))
            yield batch

    def __len__(self):
        return self.num_batches

train_df = df[df.split == 'train'].reset_index(drop=True)
val_df = df[df.split == 'val'].reset_index(drop=True)

# Fail-fast diagnostics. If you see these fire, your Drive upload is missing
# training scenes — only the held-out scene is present. Add more scene_NNN/
# directories to your Drive MTMC_Tracking_2024/ folder and re-run Phase 3.
P, K = 8, 4
n_train_rows = len(train_df)
n_train_ids = train_df['person_id'].nunique() if n_train_rows else 0
n_val_rows = len(val_df)

if n_train_rows == 0:
    raise RuntimeError(
        'train_df is empty. Your dataset only contains the held-out scene '
        f"(HOLDOUT_SCENE='{HOLDOUT_SCENE}'). Upload at least one other "
        'scene_NNN/ directory to your Drive folder and re-run Phase 3.'
    )
if n_train_ids < P:
    raise RuntimeError(
        f'train_df has only {n_train_ids} unique person_ids; PKSampler needs '
        f'at least P={P}. Upload more training scenes (target: 50-200 unique IDs '
        'across all training scenes; see spec section "Data sourcing").'
    )
pid_counts = train_df.groupby('person_id').size()
n_train_ids_keepable = int((pid_counts >= K).sum())
if n_train_ids_keepable < P:
    raise RuntimeError(
        f'Only {n_train_ids_keepable} train IDs have >= K={K} crops; PKSampler '
        f'requires >= P={P}. Either lower K (loses triplet hard mining quality) '
        'or lower FRAME_STEP in Phase 3 to get more crops per ID.'
    )
if n_val_rows == 0:
    print(f"WARNING: val_df is empty (HOLDOUT_SCENE='{HOLDOUT_SCENE}' not in manifest). "
          'Eval cells will be skipped.')

train_ds = ReIDCropDataset(train_df, TRAIN_TF)
val_ds = ReIDCropDataset(val_df, EVAL_TF) if n_val_rows else None

BATCH = P * K
sampler = PKSampler(train_ds.labels, P=P, K=K)
train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True) if val_ds else None

NUM_TRAIN_IDS = len(set(train_ds.labels))
print(f'train crops={len(train_ds)} ids={NUM_TRAIN_IDS} (keepable >=K: {n_train_ids_keepable})')
print(f'val   crops={len(val_ds) if val_ds else 0} ids={len(set(val_ds.labels)) if val_ds else 0}')
print(f'batches/epoch={len(sampler)}  |  P={P} K={K} batch={BATCH}')

## Phase 5 — Model assembly + freeze policy

- Backbone = SOLIDER Swin-Small loaded from base checkpoint (so we start from MSMT17 priors, not scratch).
- Freeze `base.stages.0/1/2.*`; train `base.stages.3.*`, `base.norm*`, `bottleneck.*`, and the new ID head.
- The ID head is discarded at inference — the pipeline only consumes the BN-neck output.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from aic24_nvidia.models.solider import load_solider_swin_small, SOLIDER_SIZE

class ReIDTrainer(nn.Module):
    def __init__(self, base_ckpt, num_ids):
        super().__init__()
        # Wrapper module already includes Swin backbone + BN bottleneck (NECK_FEAT='after').
        self.backbone = load_solider_swin_small(base_ckpt)
        # We also need the pre-BN feature for the triplet/center loss
        # (BoT recipe: ID loss on BN feat, metric losses on pre-BN feat).
        self.id_head = nn.Linear(768, num_ids, bias=False)
        nn.init.normal_(self.id_head.weight, std=0.001)

    def forward(self, x):
        global_feat, _ = self.backbone.base(x)
        bn_feat = self.backbone.bottleneck(global_feat)
        logits = self.id_head(bn_feat)
        return logits, global_feat, bn_feat

    @torch.no_grad()
    def embed(self, x):
        return self.backbone(x)   # BN-neck output, matches inference

model = ReIDTrainer('weights/solider_swin_small.pth', NUM_TRAIN_IDS).cuda()

# Freeze policy: train stages.3 + final norm + bottleneck (and id_head trivially).
# Swin output norms are named norm0..norm3; only norm3 belongs to the unfrozen stage.
TRAINABLE_BACKBONE_KEYS = ('base.stages.3', 'base.norm3', 'bottleneck')
trainable, frozen = 0, 0
for name, p in model.named_parameters():
    if name.startswith('id_head') or any(k in name for k in TRAINABLE_BACKBONE_KEYS):
        p.requires_grad = True
        trainable += p.numel()
    else:
        p.requires_grad = False
        frozen += p.numel()
print(f'trainable params: {trainable/1e6:.2f}M  |  frozen: {frozen/1e6:.2f}M  |  ratio: {trainable/(trainable+frozen):.1%}')

# Quick forward smoke test.
model.train()
_x, _y, _ = next(iter(train_loader))
_l, _gf, _bf = model(_x.cuda())
print('logits', _l.shape, '| global', _gf.shape, '| bn', _bf.shape)
del _x, _y, _l, _gf, _bf

## Phase 6 — Losses (ID + hard-triplet + center)

Bag-of-Tricks recipe (Luo et al., CVPR 2019), matching what SOLIDER's own checkpoint was trained with.

In [ ]:
def euclidean_dist_sq(x, y):
    # (n, m) pairwise squared distances
    xx = (x * x).sum(1, keepdim=True)
    yy = (y * y).sum(1, keepdim=True)
    return (xx + yy.t() - 2 * x @ y.t()).clamp(min=1e-12)

class HardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
    def forward(self, feat, labels):
        n = feat.size(0)
        d = euclidean_dist_sq(feat, feat).sqrt()
        same = labels.unsqueeze(0) == labels.unsqueeze(1)
        diff = ~same
        # hardest positive: max over same; hardest negative: min over diff.
        d_pos = (d * same.float()).max(dim=1).values
        d_neg_masked = d.clone()
        d_neg_masked[~diff] = float('inf')
        d_neg = d_neg_masked.min(dim=1).values
        return F.relu(d_pos - d_neg + self.margin).mean()

class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim=768):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim) * 0.01)
    def forward(self, feat, labels):
        c = self.centers[labels]
        return ((feat - c) ** 2).sum(dim=1).mean() * 0.5

ce = nn.CrossEntropyLoss(label_smoothing=0.1)
triplet = HardTripletLoss(margin=0.3).cuda()
center = CenterLoss(NUM_TRAIN_IDS).cuda()

LAMBDA_ID, LAMBDA_TRI, LAMBDA_CEN = 1.0, 1.0, 5e-4
print('losses ready.')

## Phase 7 — Training loop

- Optimizer: AdamW, weight decay 1e-4.
- LRs: id_head 1e-4, unfrozen backbone block 1e-5. Center loss has its own SGD (standard for BoT recipe).
- Schedule: 5-epoch linear warmup → cosine decay over the rest. Total 25 epochs (tune to your time budget).
- Save best-by-val-Rank-1.

Expected wall time: T4 ~2h for 25 epochs, A100 ~30 min. Cut `EPOCHS` if needed.

In [ ]:
from torch.optim import AdamW, SGD
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

EPOCHS = 25
WARMUP = 5

backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('id_head')]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('id_head')]

opt = AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': head_params, 'lr': 1e-4},
], weight_decay=1e-4)
opt_center = SGD(center.parameters(), lr=0.5)

sched = SequentialLR(
    opt,
    schedulers=[
        LinearLR(opt, start_factor=0.01, total_iters=WARMUP),
        CosineAnnealingLR(opt, T_max=EPOCHS - WARMUP),
    ],
    milestones=[WARMUP],
)

scaler = torch.amp.GradScaler('cuda')
print(f'optimizer ready. Epochs={EPOCHS} warmup={WARMUP}.')

In [ ]:
@torch.no_grad()
def embed_loader(model, loader):
    model.eval()
    feats, labels, cams = [], [], []
    for x, y, cam in loader:
        x = x.cuda(non_blocking=True)
        with torch.amp.autocast('cuda'):
            f = model.embed(x)
        feats.append(F.normalize(f.float(), dim=1).cpu())
        labels.append(torch.tensor(y))
        cams.extend(cam)
    return torch.cat(feats), torch.cat(labels), cams

def cross_cam_rank1_map(feats, labels, cams):
    """Each crop queries against the rest of the gallery, EXCLUDING its own camera."""
    feats = feats.numpy()
    labels = labels.numpy()
    cams = np.array(cams)
    sim = feats @ feats.T
    np.fill_diagonal(sim, -np.inf)
    r1_hits, aps = [], []
    for i in range(len(feats)):
        same_cam = cams == cams[i]
        sim_i = sim[i].copy()
        sim_i[same_cam] = -np.inf
        order = np.argsort(-sim_i)
        order = order[sim_i[order] > -np.inf]
        if len(order) == 0:
            continue
        rel = (labels[order] == labels[i]).astype(np.float32)
        if rel.sum() == 0:
            continue
        r1_hits.append(rel[0])
        cum_hits = np.cumsum(rel)
        precisions = cum_hits / (np.arange(len(rel)) + 1)
        ap = (precisions * rel).sum() / rel.sum()
        aps.append(ap)
    return float(np.mean(r1_hits)), float(np.mean(aps))

def quick_eval():
    feats, labels, cams = embed_loader(model, val_loader)
    r1, mAP = cross_cam_rank1_map(feats, labels, cams)
    return r1, mAP

# baseline eval BEFORE training so we have a sanity floor.
r1_0, map_0 = quick_eval()
print(f'pre-train (base SOLIDER): cross-cam Rank-1={r1_0:.4f}  mAP={map_0:.4f}')

In [ ]:
from pathlib import Path
import time

CKPT_DIR = Path('weights')
CKPT_DIR.mkdir(exist_ok=True)
BEST_PATH = CKPT_DIR / 'solider_swin_small_warehouse_ft_best.pth'
LAST_PATH = CKPT_DIR / 'solider_swin_small_warehouse_ft_last.pth'

def save_inference_ckpt(path):
    """Save state_dict in the same shape load_solider_swin_small expects.
    Keys: base.*, bottleneck.*  — id_head is dropped (inference-incompatible)."""
    sd = model.backbone.state_dict()
    torch.save(sd, path)

best_r1 = r1_0
history = []
for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    losses = {'total': 0.0, 'id': 0.0, 'tri': 0.0, 'cen': 0.0}
    correct = total = 0
    for x, y, _cam in train_loader:
        x = x.cuda(non_blocking=True)
        y = torch.tensor(y).cuda(non_blocking=True) if not isinstance(y, torch.Tensor) else y.cuda(non_blocking=True)
        opt.zero_grad(set_to_none=True)
        opt_center.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits, gf, bf = model(x)
            l_id = ce(logits, y)
            l_tri = triplet(gf.float(), y)
            l_cen = center(gf.float(), y)
            loss = LAMBDA_ID * l_id + LAMBDA_TRI * l_tri + LAMBDA_CEN * l_cen
        scaler.scale(loss).backward()
        scaler.step(opt)
        # scale center-loss gradient back up: standard BoT trick.
        for p in center.parameters():
            if p.grad is not None and LAMBDA_CEN > 0:
                p.grad.data.mul_(1.0 / LAMBDA_CEN)
        opt_center.step()
        scaler.update()
        losses['total'] += loss.item()
        losses['id'] += l_id.item()
        losses['tri'] += l_tri.item()
        losses['cen'] += l_cen.item()
        with torch.no_grad():
            correct += (logits.argmax(1) == y).sum().item()
            total += y.numel()
    sched.step()
    n = max(1, len(train_loader))
    acc = correct / max(1, total)
    r1, mAP = quick_eval()
    dt = time.time() - t0
    print(f'ep{epoch+1:02d}/{EPOCHS}  loss={losses["total"]/n:.3f}  id={losses["id"]/n:.3f}  tri={losses["tri"]/n:.3f}  cen={losses["cen"]/n:.4f}  acc={acc:.3f}  R1={r1:.4f}  mAP={mAP:.4f}  ({dt:.1f}s)')
    history.append({'epoch': epoch+1, 'loss': losses['total']/n, 'acc': acc, 'rank1': r1, 'mAP': mAP})
    save_inference_ckpt(LAST_PATH)
    if r1 > best_r1:
        best_r1 = r1
        save_inference_ckpt(BEST_PATH)
        print(f'  → new best, saved to {BEST_PATH}')

print(f'\nDone. Best cross-cam Rank-1 = {best_r1:.4f}  (pre-train was {r1_0:.4f})')

## Phase 8 — Evaluation

Critical decision metrics (from spec §Evaluation):
1. Cross-camera Rank-1 / Rank-5 / mAP on scene_044.
2. Per-camera-pair cosine margin (same-id vs diff-id), 7×7 matrix.
3. **Mean same-ID cosine between back-cams (0394/0395/0396) and front-cams (0390-0393).** Target > 0.50, ideally > 0.70.

In [ ]:
# Load best checkpoint into the wrapper module (matches inference exactly).
from aic24_nvidia.models.solider import load_solider_swin_small
ft_model = load_solider_swin_small(str(BEST_PATH)).cuda()

@torch.no_grad()
def embed_dataset(m, loader):
    m.eval()
    feats, labels, cams = [], [], []
    for x, y, cam in loader:
        x = x.cuda(non_blocking=True)
        with torch.amp.autocast('cuda'):
            f = m(x)
        feats.append(F.normalize(f.float(), dim=1).cpu())
        labels.append(torch.tensor(y) if not isinstance(y, torch.Tensor) else y)
        cams.extend(cam)
    return torch.cat(feats), torch.cat(labels), cams

feats, labels, cams = embed_dataset(ft_model, val_loader)
print('val feats:', feats.shape)

In [ ]:
# Cross-camera Rank-1 / Rank-5 / mAP
import numpy as np
F_np = feats.numpy()
L = labels.numpy()
C = np.array(cams)
sim = F_np @ F_np.T
np.fill_diagonal(sim, -np.inf)

r1_hits = []; r5_hits = []; aps = []
for i in range(len(F_np)):
    s = sim[i].copy()
    s[C == C[i]] = -np.inf
    order = np.argsort(-s)
    order = order[s[order] > -np.inf]
    if len(order) == 0:
        continue
    rel = (L[order] == L[i]).astype(np.float32)
    if rel.sum() == 0:
        continue
    r1_hits.append(rel[0])
    r5_hits.append(rel[:5].max())
    cum_hits = np.cumsum(rel)
    precisions = cum_hits / (np.arange(len(rel)) + 1)
    aps.append((precisions * rel).sum() / rel.sum())
print(f'Cross-camera Rank-1 = {np.mean(r1_hits):.4f}')
print(f'Cross-camera Rank-5 = {np.mean(r5_hits):.4f}')
print(f'Cross-camera mAP    = {np.mean(aps):.4f}')

In [ ]:
# Per-camera-pair cosine margin (same-ID vs diff-ID) — 7×7 matrix.
import pandas as pd
uniq_cams = sorted(set(cams))
n_cams = len(uniq_cams)
margin = np.full((n_cams, n_cams), np.nan)
mean_same = np.full((n_cams, n_cams), np.nan)
for a, cam_a in enumerate(uniq_cams):
    ia = np.where(C == cam_a)[0]
    for b, cam_b in enumerate(uniq_cams):
        ib = np.where(C == cam_b)[0]
        if len(ia) == 0 or len(ib) == 0:
            continue
        s = F_np[ia] @ F_np[ib].T
        same = L[ia][:, None] == L[ib][None, :]
        if a == b:
            # Exclude self-self diagonal trivially.
            mask_self = np.eye(len(ia), dtype=bool)
            same = same & ~mask_self
        diff = ~same
        if same.sum() == 0 or diff.sum() == 0:
            continue
        ms = s[same].mean()
        md = s[diff].mean()
        margin[a, b] = ms - md
        mean_same[a, b] = ms
print('Same-ID mean cosine (rows=query cam, cols=gallery cam):')
print(pd.DataFrame(mean_same, index=uniq_cams, columns=uniq_cams).round(3))
print('\nMargin = same - diff:')
print(pd.DataFrame(margin, index=uniq_cams, columns=uniq_cams).round(3))

In [ ]:
# Critical decision metric: back-cam ↔ front-cam mean same-ID cosine.
BACK = {'camera_0394', 'camera_0395', 'camera_0396'}
FRONT = {'camera_0390', 'camera_0391', 'camera_0392', 'camera_0393'}

back_idx = [i for i, c in enumerate(uniq_cams) if c in BACK]
front_idx = [i for i, c in enumerate(uniq_cams) if c in FRONT]

back_front_same = np.nanmean(mean_same[np.ix_(back_idx, front_idx)])
back_front_margin = np.nanmean(margin[np.ix_(back_idx, front_idx)])
print(f'Back ↔ Front mean same-ID cosine = {back_front_same:.4f}  (target > 0.50, ideal > 0.70)')
print(f'Back ↔ Front mean margin         = {back_front_margin:.4f}  (target > 0.40)')

# Per back-cam, mean same-ID cosine vs front cams:
for bi, b in zip(back_idx, [uniq_cams[i] for i in back_idx]):
    val = np.nanmean(mean_same[bi, front_idx])
    print(f'  {b} → front cams: same-ID cos = {val:.4f}')

## Phase 9 — Save outputs to Drive

Persist the final checkpoint to Drive so it survives the Colab VM dying. The local repo only needs this single `.pth` file.

In [ ]:
import shutil
DRIVE_OUT = '/content/drive/MyDrive/aic24_reid_finetune'
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy(BEST_PATH, f'{DRIVE_OUT}/solider_swin_small_warehouse_ft.pth')
shutil.copy(LAST_PATH, f'{DRIVE_OUT}/solider_swin_small_warehouse_ft_last.pth')

# Dump training history + eval metrics for the PR.
import json
summary = {
    'pre_train': {'rank1': r1_0, 'mAP': map_0},
    'best_rank1': best_r1,
    'history': history,
    'back_front_same_id_cos': float(back_front_same),
    'back_front_margin': float(back_front_margin),
    'epochs': EPOCHS,
    'P_K': [P, K],
    'lambda_id_tri_cen': [LAMBDA_ID, LAMBDA_TRI, LAMBDA_CEN],
}
with open(f'{DRIVE_OUT}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved to {DRIVE_OUT}:')
!ls -lh "$DRIVE_OUT"

## Phase 10 — Local integration (do this back on your laptop)

1. Download the checkpoint from Drive to your local repo:
    ```bash
    cp /path/to/Drive/aic24_reid_finetune/solider_swin_small_warehouse_ft.pth \
       ~/github-repos/awl-research/aic24-nvidia/weights/
    ```
2. Add the `reid.checkpoint_path` knob (spec §Local integration → Code changes required):
    - `aic24_nvidia/config.py`: add `checkpoint_path: str = 'weights/solider_swin_small.pth'` to `ReidCfg` and parse it in `load_config`.
    - `aic24_nvidia/models/reid_solider.py`: thread `cfg.reid.checkpoint_path` through to `load_solider_swin_small(...)` in `_embed`.
    - `configs/baseline.yaml`: set `reid.checkpoint_path: weights/solider_swin_small_warehouse_ft.pth`.
3. Rebuild reid + downstream stages on the v3 baseline (~25 min) and compare:
    ```bash
    source .venv/bin/activate
    rm -rf outputs/baseline/{reid,sct,mct,evaluate}
    python pipeline.py all --config configs/baseline.yaml --run-id baseline
    ```
4. Apply the Decision Rule (spec table):
    - Back-cam global IDs ≥ 5 AND world HOTA ≥ v3 + 0.01 → **ship**: lock as v4 baseline, update `pipeline-state.md` memory.
    - Linking improved but HOTA flat → sweep `sim_th` 0.85 → 0.75 (cheap variant in `experiments/registry.yaml`).
    - Pipeline regressed → backbone forgot too much; reduce backbone LR (1e-5 → 1e-6) or freeze stage 4 too.
    - Pipeline unchanged → verify the checkpoint actually loaded (log a feature distribution); if load is fine, escalate per spec §Escalation paths.
5. Commit the notebook back to the repo:
    ```bash
    git add notebooks/reid_finetune.ipynb
    git commit -m 'feat(reid): partial fine-tune notebook for back-cam adaptation'
    ```